In [1]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re
import openpyxl
import numpy as np

# #%pip install pygwalker
# import pygwalker as pyg

## Data Importing

In [ ]:
df_sales=pd.read_excel(r"Sales.xlsx",sheet_name='Sales')
df_sales.head(5)

In [ ]:
ProductMap=r"Product_Map.xlsx"

In [ ]:
df_PM=pd.read_excel(ProductMap,sheet_name='Product Map')
df_PM.head(5)

In [ ]:
df_Exclusions=pd.read_excel(ProductMap,sheet_name='Exclusion',usecols="B:D",skiprows=1)
df_Exclusions

In [ ]:
df_Packinginfo=pd.read_excel(ProductMap,sheet_name="Type")
df_Packinginfo.head(5)

In [7]:
# Customer Segmentation
SpecificList=['Hopkins', 'BPI', 'Centric','Carter','Horizon','Cardone']
GenericList=["TRICO","Champ","Autolite","FRAM","IBI",'AVM']

In [8]:
for i in range(len(df_sales)):
    business_unit = df_sales.BusinessUnit[i]
    product_desc = df_sales.ProductDesc[i]

    if business_unit in SpecificList:
        try:
            product_name = df_PM.loc[
                df_PM['Product Description'] == product_desc, 'Product Name'
            ].values[0]
            df_sales.loc[i, 'ProductName'] = product_name
        except Exception:
            df_sales.loc[i, 'ProductName'] = "ProductNot Found in Mapping"

    elif business_unit in GenericList:
        if business_unit == 'FRAM' and 'AIR' in product_desc:
            df_sales.loc[i, 'ProductName'] = ""
        elif business_unit == 'Champ' and 'AIR' in product_desc:
            df_sales.loc[i, 'ProductName'] = "Air Filter"
        elif business_unit == 'Champ' and 'OIL' in product_desc:
            df_sales.loc[i, 'ProductName'] = "Oil Filter"
        else:
            try:
                product_name = df_PM.loc[
                    df_PM['Customer'] == business_unit, 'Product Name'
                ].values[0]
                df_sales.loc[i, 'ProductName'] = product_name
            except Exception:
                df_sales.loc[i, 'ProductName'] = "ProductNot Found in Mapping"
    else:
        df_sales.loc[i, 'ProductName'] = ""


In [ ]:
df_sales

In [10]:
df_sales=df_sales[['CustomerDesc','ProductCode', 'BusinessUnit','ProductDesc','CustomerState', 'ProductName', 'QTY SHIPPED']] #Removing unwanted columns from the original dataset
df_sales['Key']=df_sales['BusinessUnit']+df_sales['ProductName'] #Creating Key column for merging with Product Map

In [11]:
df_final=df_sales.merge(df_Packinginfo, how='left', left_on='Key', right_on='Concat', indicator=True) # merging with Product Map to get the packing info
df_final['BUCUSTOMER']=df_final['BusinessUnit']+df_final['CustomerDesc']

In [ ]:
df_final

In [13]:
df_final=df_final.merge(df_Exclusions, how='left', indicator='Source') #merginging with Exclusions to get the exclusions

In [14]:
df_final['CustomerType'] = np.where(df_final['Source']=='both', 'Exclusion', 'Inclusion')
df_final['TotalWeight']=df_final['QTY SHIPPED']*df_final['Qty']*df_final['Average weight (lb)']

In [ ]:
df_final

In [16]:
df_final=df_final[['CustomerDesc','CustomerType','BusinessUnit','ProductCode', 'ProductDesc', 'ProductName', 'Line Number','CustomerState',
       'QTY SHIPPED', 'Material Category', 'Average weight (lb)',  'Qty','TotalWeight']]

In [ ]:
df_final

In [ ]:
df_mapped = df_final.pivot_table(
    index=['CustomerType','Material Category'],
    columns=['BusinessUnit'],
    values='TotalWeight',
    aggfunc='sum'
).reset_index()

df_mapped.iloc[:, 1:] = df_mapped.iloc[:, 1:].round(1)

df_mapped


In [ ]:
FileName=r"Path/Filename.xlsx"
with pd.ExcelWriter(FileName) as writer:  # doctest: +SKIP
    df_final.to_excel(writer,index=False, sheet_name='Raw')
    df_mapped.to_excel(writer,index=False, sheet_name='Summary')

In [ ]:
# df_bpi=pd.read_csv(r"path/filename.csv")
# df_bpi

In [ ]:
# df_Framauto=pd.read_excel(r"path/filename.xlsx",sheet_name="Sheet3")
# df_Framauto

In [ ]:
df_champ=pd.read_excel(r"path/filename.xlsx")
df_champ